In [1]:
import pandas as pd
import numpy as np
import os
import joblib  # .pkl 파일을 불러오기 위해 (pickle도 가능)
from scipy.sparse import load_npz # .npz (희소 행렬) 파일을 불러오기 위해
from sklearn.metrics.pairwise import cosine_similarity

print("필요한 라이브러리를 모두 임포트했습니다.")

############################################################
# 1. 파일 경로 설정 및 데이터 로드
############################################################

# --- 경로 설정 ---
# 현재 .ipynb 파일이 있는 'notebooks' 폴더 기준으로 상위 폴더(.)가 기준
BASE_DIR = '../'
# 원본 데이터가 있는 'data' 폴더 경로
DATA_DIR = os.path.join(BASE_DIR, 'data')
# 전처리/임베딩 결과물을 저장할 'output' 폴더 경로
OUTPUT_DIR = os.path.join(BASE_DIR, 'output')

# 만약 'output' 폴더가 존재하지 않는다면,
# exist_ok=True 옵션으로 오류 없이 폴더를 생성
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 설정된 경로 확인
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")
# (필수 파일 1) 전처리된 메타 데이터 로드
meta_path = os.path.join(OUTPUT_DIR, 'preprocessed_metadata.pkl')
metadata_df = pd.read_pickle(meta_path)

# (필수 파일 2) TF-IDF 임베딩 행렬 로드
matrix_path = os.path.join(OUTPUT_DIR, 'tfidf_embedding_matrix.npz')
tfidf_matrix = load_npz(matrix_path)

############################################################
# 💡💡💡 [수정된 부분] 💡💡💡
# metadata_df의 인덱스를 0부터 시작하도록 리셋하여
# tfidf_matrix의 행 순서(0, 1, 2...)와 일치시킵니다.
metadata_df = metadata_df.reset_index(drop=True)
############################################################


# (선택적) 디버깅을 위해 두 파일의 행 개수가 일치하는지 확인
print(f"메타데이터 로드 완료. (총 {len(metadata_df)}개 아이템)")
print(f"TF-IDF 행렬 로드 완료. (형태: {tfidf_matrix.shape})")

if metadata_df.shape[0] != tfidf_matrix.shape[0]:
    print("!!!경고!!! 메타데이터와 TF-IDF 행렬의 아이템 개수가 일치하지 않습니다.")


############################################################
# 2. 'asin' <-> '행렬 인덱스' 매핑 생성
############################################################

# 이제 metadata_df.index는 [0, 1, 2, ..., N-1] 입니다.
# 'asin'을 key로, metadata_df의 인덱스(0, 1, 2...)를 value로 하는
# Series를 만듭니다.
asin_to_index = pd.Series(metadata_df.index, index=metadata_df['asin'])

############################################################
# 3. 콘텐츠 기반 추천 함수 정의 (핵심 역할)
############################################################

def get_content_based_recommendations(target_asin, N=10): # 기본값 N = 10
    """
    특정 아이템(asin)과 텍스트(TF-IDF)가 유사한 Top-N개 아이템을 반환합니다.
    
    Args:
        target_asin (str): 기준이 되는 아이템의 asin
        N (int): 추천할 아이템 개수
        
    Returns:
        pandas.DataFrame: 추천 아이템 리스트 ('asin', 'title_str', 'similarity')
    """
    
    # 1. target_asin이 메타데이터에 있는지 확인
    if target_asin not in asin_to_index:
        return f"'{target_asin}'에 해당하는 아이템을 찾을 수 없습니다."
        
    # 2. target_asin을 tfidf_matrix의 행 인덱스(숫자)로 변환
    item_index = asin_to_index[target_asin]
    
    # 3. 해당 아이템의 TF-IDF 벡터를 가져옴
    # (1, n_features) 형태의 희소 행렬
    target_vector = tfidf_matrix[item_index] 
    
    # 4. 이 벡터와 *전체* 아이템 벡터 간의 코사인 유사도 계산
    # (1, n_features)와 (n_items, n_features) 계산 -> (1, n_items) 결과 반환
    sim_scores = cosine_similarity(target_vector, tfidf_matrix)
    
    # 5. 계산된 유사도 점수를 1차원 배열로 변환하고, 
    # (점수, 아이템 인덱스) 튜플 리스트로 만듦
    sim_scores_list = sim_scores.flatten()
    
    # 6. 유사도가 높은 순서대로 아이템 인덱스를 정렬 (내림차순)
    # argsort는 '값'이 아니라 '인덱스'를 정렬하여 반환
    # [::-1]을 붙여 내림차순으로 변경
    similar_item_indices = np.argsort(sim_scores_list)[::-1]
    
    # 7. Top N+1 개 추출 (자기 자신 포함)
    # (맨 처음[0]은 자기 자신이므로 유사도 1.0)
    top_indices = similar_item_indices[1 : N+1] # 자기 자신(0번) 제외
    
    # 8. 인덱스를 실제 asin과 title로 변환
    recommended_items = metadata_df.iloc[top_indices][['asin', 'title_str']].copy()
    
    # 9. 유사도 점수도 함께 추가
    recommended_items['similarity'] = sim_scores_list[top_indices]
    
    return recommended_items


############################################################
# 4. 함수 사용 예시
############################################################

# 'popularity_model_top_items.csv'에 있던 'A Charlie Brown Christmas' 앨범
TEST_ASIN = 'B000000XDJ' 

#  여기 N 값만 바꾸면 함수 호출과 print 문에 모두 적용됩니다.
N = 10

recommendations_df = get_content_based_recommendations(TEST_ASIN, N)

print(f"\n--- '{TEST_ASIN}'와 유사한 아이템 Top {N}개---")

# 결과 출력 (Jupyter Notebook에서는 print() 대신 바로 recommendations_df)
print(recommendations_df.to_markdown(index=False))

필요한 라이브러리를 모두 임포트했습니다.
Data directory: ../data
Output directory: ../output
메타데이터 로드 완료. (총 112942개 아이템)
TF-IDF 행렬 로드 완료. (형태: (112942, 20000))

--- 'B000000XDJ'와 유사한 아이템 Top 10개---
| asin       | title_str                                 |   similarity |
|:-----------|:------------------------------------------|-------------:|
| B000ICLSMY | A Charlie Brown Christmas                 |     0.872145 |
| B006U1S9CE | Linus & Lucy- The Music of Vince Guaraldi |     0.295628 |
| B08CPHFTTG | A Charlie Brown Christmas                 |     0.287484 |
| B000A2GX4A | 40 Years: A Charlie Brown Christmas       |     0.27926  |
| B00IXPGR6K | A Boy Named Charlie Brown                 |     0.246807 |
| B000000XD9 | Vince Guaraldi - Greatest Hits            |     0.242887 |
| B000009OG6 | Charlie Brown's Holiday Hits              |     0.226449 |
| B000MJFK3G | The Perfect Christmas: Holiday Music 2006 |     0.224287 |
| B00004U9MM | Charlie Brown Christmas                   |     0.217836 |
| B09